In [ ]:
import os
import csv
import torch
import pandas as pd
from tqdm import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

def generate_dpo_dataset_batched(
    input_csv_path: str,
    model_name: str = "Qwen/Qwen2.5-7B-Instruct",
    batch_size: int = 4,
    max_new_tokens: int = 180,
    temperature: float = 0.7
):
    safe_model_name = model_name.split("/")[-1]
    output_csv_path = f"final_pairs_dpo_{safe_model_name}.csv"
    
    print(f"Loading dataset from: {input_csv_path}")
    df = pd.read_csv(input_csv_path)
    
    if not {"question", "answer"}.issubset(df.columns):
        raise ValueError("Input CSV must contain 'question' and 'answer' columns.")

    start_index = 0
    if os.path.exists(output_csv_path):
        existing_df = pd.read_csv(output_csv_path)
        start_index = len(existing_df)
        print(f"Found existing output file. Resuming from index: {start_index} out of {len(df)}")
        if start_index >= len(df):
            print("Dataset already fully processed. Exiting.")
            return
    else:
        with open(output_csv_path, mode='w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(["question", "answer_w", "answer_l"])
        print(f"Created new output file: {output_csv_path}")

    print(f"\nLoading model: {model_name} in 4-bit precision...")
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left" 

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        torch_dtype=compute_dtype,
        device_map={"": 0},
        attn_implementation="flash_attention_2" 
    )
    model.eval()

    print(f"\nStarting batched generation (Batch size: {batch_size})...")
    
    remaining_df = df.iloc[start_index:]
    
    for i in tqdm(range(0, len(remaining_df), batch_size), desc="Processing Batches"):
        batch = remaining_df.iloc[i : i + batch_size]
        
        questions = batch["question"].astype(str).str.strip().tolist()
        winning_answers = batch["answer"].astype(str).tolist()
        
        system_instruction = (
            "You are an objective analytical system. Answer the question directly in a "
            "single, concise paragraph of one to two sentences. Write exclusively in plain text. "
            "You are strictly forbidden from using bullet points, numbered lists, markdown formatting, "
            "or introductory filler phrases. Provide a direct, declarative explanation."
        )

        prompts = []
        for q in questions:
            messages = [
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": q}
            ]
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            prompts.append(prompt)
            
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True,
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            
        prompt_lengths = inputs["input_ids"].shape[1]
        generated_tokens = outputs[:, prompt_lengths:]
        
        rejected_answers = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
        
        with open(output_csv_path, mode='a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            for q, w_ans, l_ans in zip(questions, winning_answers, rejected_answers):
                writer.writerow([q, w_ans, l_ans.strip()])

    print(f"\nProcessing complete! DPO dataset saved to {output_csv_path}")

if __name__ == "__main__":
    DATA_PATH = "final_pairs.csv" 
    TARGET_MODEL = "Qwen/Qwen2.5-7B-Instruct" 
    
    generate_dpo_dataset_batched(
        input_csv_path=DATA_PATH,
        model_name=TARGET_MODEL,
        batch_size=8, 
        temperature=0.7 
    )

Loading dataset from: final_pairs.csv
Created new output file: final_pairs_dpo_Qwen2.5-7B-Instruct.csv

Loading model: Qwen/Qwen2.5-7B-Instruct in 4-bit precision...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/venv/main/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Starting batched generation (Batch size: 8)...


Processing Batches: 100%|██████████| 555/555 [30:36<00:00,  3.31s/it]


Processing complete! DPO dataset saved to final_pairs_dpo_Qwen2.5-7B-Instruct.csv
